# SWOT RiverSP demo interactive annotation

This notebook interactively reviews all test-level candidate cases in the included demo data. It displays the candidate location on a satellite map and plots the local multi-date WSE profile. Labels are saved to a new CSV with an autosave backup and a resumable cursor.

Run the cells from top to bottom. The notebook reads its inputs from this repository’s `data/demo_data/`. The satellite basemap is loaded online.


In [1]:
from pathlib import Path
import html
import shutil

import folium
import ipywidgets as widgets
import matplotlib.colors as mcolors
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from IPython.display import clear_output, display

# Locate this repository from the Jupyter kernel's current directory.
PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "data" / "demo_data" / "demo_eu_test_raw.csv").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("Could not locate this repository's data/demo_data directory. Start Jupyter inside the repository.")
    PROJECT_ROOT = PROJECT_ROOT.parent
DEMO_DIR = PROJECT_ROOT / "data" / "demo_data"
CANDIDATE_PATH = DEMO_DIR / "demo_eu_test_rf_predictions.csv"
NODE_PATH = DEMO_DIR / "demo_eu_test_raw.csv"
GEOMETRY_PATH = DEMO_DIR / "demo_eu_test_node_geometry.csv"
SAVE_PATH = DEMO_DIR / "demo_eu_test_candidates_interactive_ANNOTATED.csv"
BACKUP_PATH = SAVE_PATH.with_name(SAVE_PATH.stem + "_AUTOSAVE_BACKUP.csv")
CURSOR_COL = "__review_cursor__"
TYPE_GROUP_N = ["waterfall", "rapid"]
TYPE_GROUP_A = ["dam", "low head", "bridge"]

df_review = pd.read_csv(CANDIDATE_PATH, encoding="utf-8-sig")
df_review["node_id"] = pd.to_numeric(df_review["node_id"], errors="raise").astype("int64")
for col in ["T/F", "N/A", "type", "note2", CURSOR_COL]:
    if col not in df_review.columns:
        df_review[col] = ""
    df_review[col] = df_review[col].fillna("").astype(str).replace("nan", "").str.strip()

df_node = pd.read_csv(NODE_PATH, encoding="utf-8-sig")
df_node["node_id"] = pd.to_numeric(df_node["node_id"], errors="raise").astype("int64")
df_node["reach_id"] = df_node["node_id"] // 10000 * 10 + df_node["node_id"] % 10
df_node["date"] = pd.to_datetime(df_node["time"], unit="s", origin="2000-01-01").dt.floor("D")
df_geo = pd.read_csv(GEOMETRY_PATH, encoding="utf-8-sig", dtype={"node_id": "int64"})
df_geo = df_geo.dropna(subset=["lon", "lat"]).drop_duplicates("node_id", keep="first")
node_lookup = df_geo.set_index("node_id")[["lon", "lat"]]
df_node = df_node.join(node_lookup, on="node_id", how="left")
if df_review["node_id"].isin(node_lookup.index).sum() != len(df_review):
    missing = sorted(set(df_review["node_id"]) - set(node_lookup.index))
    raise ValueError(f"Candidate nodes are missing coordinates: {missing[:10]}")

print(f"Loaded {len(df_review):,} test candidates and {df_node['node_id'].nunique():,} test nodes.")
print(f"Labels will be saved to: {SAVE_PATH}")


Loaded 139 test candidates and 6,118 test nodes.
Labels will be saved to: D:\github\SWOT_RiverSP_Obstruction\data\demo_data\demo_eu_test_candidates_interactive_ANNOTATED.csv


In [2]:
def _rgba_to_hex(rgba):
    return mcolors.to_hex(rgba, keep_alpha=False)

def _make_node_color_map(node_ids):
    node_ids = np.array(sorted(map(np.int64, node_ids)))
    colors = plt.cm.tab20(np.linspace(0, 1, len(node_ids)))
    return {np.int64(n): _rgba_to_hex(c) for n, c in zip(node_ids, colors)}

def plot_satellite_map(df_points, node_colors_hex, target_node_id, zoom=16):
    if df_points.empty:
        raise ValueError("No node coordinates are available for the map.")
    m = folium.Map(
        location=[float(df_points["lat"].mean()), float(df_points["lon"].mean())],
        zoom_start=zoom,
        tiles="http://mt0.google.com/vt/lyrs=s&hl=en&x={x}&y={y}&z={z}",
        attr="Google",
        control_scale=True,
    )
    points = df_points.sort_values("node_id")
    folium.PolyLine(points[["lat", "lon"]].values.tolist(), weight=2, opacity=0.7).add_to(m)
    for _, row in points.iterrows():
        node_id = np.int64(row["node_id"])
        folium.CircleMarker(
            location=[float(row["lat"]), float(row["lon"])],
            radius=7 if node_id == np.int64(target_node_id) else 5,
            weight=3 if node_id == np.int64(target_node_id) else 2,
            color="white",
            fill=True,
            fill_color=node_colors_hex[node_id],
            fill_opacity=0.95,
            popup=f"node_id: {node_id}<br>lon: {row['lon']:.6f}<br>lat: {row['lat']:.6f}",
        ).add_to(m)
    return m

def plot_3d_profile(df_profile, view_elev=30, view_azim=110):
    d = df_profile.sort_values(["date", "node_id"]).copy()
    node_ids = np.sort(d["node_id"].unique())
    node_to_x = {np.int64(node): i for i, node in enumerate(node_ids)}
    d["x"] = d["node_id"].map(node_to_x).astype(float)
    d["date_num"] = mdates.date2num(pd.to_datetime(d["date"]).to_numpy())
    colors = _make_node_color_map(node_ids)
    scatter = go.Scatter3d(
        x=d["x"], y=-d["date_num"], z=d["wse"], mode="markers",
        marker=dict(size=3, color=[colors[np.int64(n)] for n in d["node_id"]], opacity=0.9),
        customdata=np.stack([d["node_id"].to_numpy(), d["date"].dt.strftime("%Y-%m-%d").to_numpy()], axis=1),
        hovertemplate="node_id=%{customdata[0]}<br>date=%{customdata[1]}<br>WSE=%{z:.3f} m<extra></extra>",
        name="SWOT observations",
    )
    lines = []
    for date, group in d.groupby("date"):
        group = group.sort_values("node_id")
        y = -float(mdates.date2num(pd.Timestamp(date).to_pydatetime()))
        lines.append(go.Scatter3d(x=group["x"], y=np.full(len(group), y), z=group["wse"], mode="lines", line=dict(width=5), opacity=0.55, hoverinfo="skip", showlegend=False))
    median = d.groupby("node_id", as_index=False)["wse"].median().sort_values("node_id")
    x_nodes = np.arange(len(node_ids), dtype=float)
    dates = np.sort(d["date"].unique())
    date_nums = mdates.date2num(pd.to_datetime(dates).to_pydatetime())
    xx, yy = np.meshgrid(x_nodes, -date_nums)
    zz = np.tile(median["wse"].to_numpy(), (len(dates), 1))
    surface = go.Surface(x=xx, y=yy, z=zz, opacity=0.3, showscale=False, hoverinfo="skip", name="Median profile")
    date_idx = np.linspace(0, len(dates) - 1, min(8, len(dates))).astype(int)
    node_idx = np.linspace(0, len(node_ids) - 1, min(11, len(node_ids))).astype(int)
    fig = go.Figure(data=[surface, *lines, scatter])
    fig.update_layout(
        height=650, title="Local WSE profile across nodes and dates",
        scene=dict(
            xaxis=dict(title="node_id", tickmode="array", tickvals=node_idx, ticktext=[str(node_ids[i]) for i in node_idx]),
            yaxis=dict(title="date", tickmode="array", tickvals=-date_nums[date_idx], ticktext=[pd.Timestamp(dates[i]).strftime("%Y-%m-%d") for i in date_idx]),
            zaxis=dict(title="WSE (m)"), aspectmode="cube",
        ),
        scene_camera=dict(eye=dict(x=1.4, y=-1.8, z=0.9)), margin=dict(l=0, r=0, t=40, b=0),
    )
    return fig

def draw_case(node_id, window=5, zoom=16):
    node_id = np.int64(node_id)
    reach_id = node_id // 10000 * 10 + node_id % 10
    reach = df_node.loc[df_node["reach_id"] == reach_id]
    unique_nodes = np.sort(reach["node_id"].unique())
    if node_id not in unique_nodes:
        raise ValueError(f"Candidate node {node_id} is absent from the test observations.")
    position = np.flatnonzero(unique_nodes == node_id)[0]
    local_nodes = unique_nodes[max(0, position-window):min(len(unique_nodes), position+window+1)]
    profile = reach.loc[reach["node_id"].isin(local_nodes)].copy()
    points = profile.sort_values("node_id").drop_duplicates("node_id")[["node_id", "lon", "lat"]]
    color_map = _make_node_color_map(local_nodes)
    target = node_lookup.loc[node_id]
    display(widgets.HTML(
        f"<b>node_id</b>: {node_id} &nbsp; <b>reach_id</b>: {reach_id} &nbsp; "
        f"<b>lon, lat</b>: {target['lon']:.6f}, {target['lat']:.6f} &nbsp; "
        f"<b>change_size</b>: {html.escape(str(df_review.iloc[cur].get('change_size', '')))} &nbsp; "
        f"<b>RF probability</b>: {html.escape(str(round(float(df_review.iloc[cur].get('pred_prob', np.nan)), 3)))}"
    ))
    display(plot_satellite_map(points, color_map, node_id, zoom=zoom))
    display(plot_3d_profile(profile))


In [3]:
# Label form and resumable review position. Type has no '?' or blank button.
w_tf = widgets.RadioButtons(options=[("Y", "Y"), ("N", "N")], description="T/F:", value=None, layout=widgets.Layout(width="180px"))
w_na = widgets.RadioButtons(options=[("N", "N"), ("A", "A")], description="N/A:", value=None, layout=widgets.Layout(width="180px"))
w_type = widgets.ToggleButtons(options=[], description="Type:", value=None, layout=widgets.Layout(width="700px", flex_flow="row wrap"))
w_note = widgets.Textarea(value="", description="Note:", placeholder="Optional note", layout=widgets.Layout(width="700px", height="90px"))
viewer = widgets.Output(layout={"border": "1px solid #ddd", "padding": "8px"})
progress = widgets.IntProgress(value=0, min=0, max=len(df_review), description="Case:")
status = widgets.HTML()
notice = widgets.HTML()
btn_save_next = widgets.Button(description="Save & Next", button_style="success")
btn_save = widgets.Button(description="Save (stay)", button_style="info")
btn_skip = widgets.Button(description="Skip", button_style="warning")
btn_back = widgets.Button(description="Back")

def set_type_options(group):
    options = TYPE_GROUP_N if group == "N" else TYPE_GROUP_A if group == "A" else []
    old = w_type.value
    w_type.options = options
    w_type.value = old if old in options else None

def apply_label_logic(*_):
    if w_tf.value == "N":
        w_na.value = None
        w_na.disabled = True
        set_type_options(None)
        w_type.disabled = True
    elif w_tf.value == "Y":
        w_na.disabled = False
        if w_na.value in ("N", "A"):
            set_type_options(w_na.value)
            w_type.disabled = False
        else:
            set_type_options(None)
            w_type.disabled = True
    else:
        w_na.value = None
        w_na.disabled = True
        set_type_options(None)
        w_type.disabled = True

w_tf.observe(apply_label_logic, names="value")
w_na.observe(apply_label_logic, names="value")

def is_complete(row):
    tf = str(row.get("T/F", "")).strip()
    if tf == "N":
        return True
    if tf != "Y":
        return False
    na = str(row.get("N/A", "")).strip()
    allowed = TYPE_GROUP_N if na == "N" else TYPE_GROUP_A if na == "A" else []
    return na in ("N", "A") and str(row.get("type", "")).strip() in allowed

def validate_form():
    if w_tf.value not in ("Y", "N"):
        raise ValueError("Choose T/F before saving.")
    if w_tf.value == "Y":
        if w_na.value not in ("N", "A"):
            raise ValueError("Choose N/A before saving a positive case.")
        allowed = TYPE_GROUP_N if w_na.value == "N" else TYPE_GROUP_A
        if w_type.value not in allowed:
            raise ValueError("Choose a Type before saving a positive case.")

def save_form():
    validate_form()
    df_review.at[cur, "T/F"] = w_tf.value
    df_review.at[cur, "N/A"] = w_na.value if w_tf.value == "Y" else ""
    df_review.at[cur, "type"] = w_type.value if w_tf.value == "Y" else ""
    df_review.at[cur, "note2"] = w_note.value.strip()
    save_now()

def save_now():
    df_review[CURSOR_COL] = "" if cur is None else str(int(cur))
    temp = SAVE_PATH.with_name(SAVE_PATH.stem + "_TEMP_WRITE.csv")
    df_review.to_csv(temp, index=False, encoding="utf-8-sig")
    if SAVE_PATH.exists():
        shutil.copy2(SAVE_PATH, BACKUP_PATH)
    temp.replace(SAVE_PATH)

def next_unlabeled(start):
    for offset in range(1, len(df_review) + 1):
        idx = (start + offset) % len(df_review)
        if not is_complete(df_review.iloc[idx]):
            return idx
    return None

def current_index():
    values = pd.to_numeric(df_review[CURSOR_COL], errors="coerce").dropna()
    if values.empty:
        return 0
    return int(np.clip(values.iloc[0], 0, len(df_review) - 1))

cur = current_index()
if len(df_review) and df_review.apply(is_complete, axis=1).all():
    cur = None
elif len(df_review) and is_complete(df_review.iloc[cur]):
    cur = next_unlabeled(cur)


In [4]:
def render_current():
    global cur
    if not len(df_review):
        status.value = "No test candidates are available."
        return
    done = int(df_review.apply(is_complete, axis=1).sum())
    if cur is None:
        progress.value = len(df_review)
        status.value = f"All {done}/{len(df_review)} cases are annotated."
        with viewer:
            clear_output(wait=True)
            print("Review complete. Use Back to revisit the last case.")
        btn_save_next.disabled = True
        btn_save.disabled = True
        btn_skip.disabled = True
        return
    btn_save_next.disabled = False
    btn_save.disabled = False
    btn_skip.disabled = False
    row = df_review.iloc[cur]
    node_id = int(row["node_id"])
    progress.value = cur + 1
    status.value = f"<b>Position:</b> {cur+1}/{len(df_review)} &nbsp; | &nbsp; <b>Annotated:</b> {done}/{len(df_review)}"
    w_tf.value = str(row["T/F"]).strip() if str(row["T/F"]).strip() in ("Y", "N") else None
    w_na.value = str(row["N/A"]).strip() if str(row["N/A"]).strip() in ("N", "A") else None
    apply_label_logic()
    current_type = str(row["type"]).strip()
    w_type.value = current_type if current_type in list(w_type.options) else None
    w_note.value = str(row["note2"]) if str(row["note2"]) != "nan" else ""
    with viewer:
        clear_output(wait=True)
        print(f"===== {cur+1}/{len(df_review)} | row={cur} | node_id={node_id} =====")
        draw_case(node_id, window=5, zoom=16)

def show_error(exc):
    notice.value = f"<span style='color:#b00020'>{html.escape(str(exc))}</span>"

def on_save_next(_):
    global cur
    try:
        save_form()
        nxt = next_unlabeled(cur)
        cur = nxt
        if cur is None:
            df_review[CURSOR_COL] = ""
            save_now()
        render_current()
        notice.value = "Saved. Moved to the next unlabeled case."
    except Exception as exc:
        show_error(exc)

def on_save(_):
    try:
        save_form()
        render_current()
        notice.value = "Saved this case."
    except Exception as exc:
        show_error(exc)

def on_skip(_):
    global cur
    if not len(df_review):
        return
    cur = next_unlabeled(cur)
    if cur is None:
        df_review[CURSOR_COL] = ""
        save_now()
        render_current()
        notice.value = "All cases are annotated."
        return
    df_review[CURSOR_COL] = str(int(cur))
    save_now()
    render_current()
    notice.value = "Skipped without saving form changes; moved to the next unlabeled case."

def on_back(_):
    global cur
    if len(df_review):
        cur = (len(df_review) - 1 if cur is None else max(0, cur - 1))
        df_review[CURSOR_COL] = str(int(cur))
        save_now()
        render_current()
        notice.value = "Moved back one case."

btn_save_next.on_click(on_save_next)
btn_save.on_click(on_save)
btn_skip.on_click(on_skip)
btn_back.on_click(on_back)
form = widgets.VBox([
    widgets.HBox([progress, status]),
    viewer,
    widgets.HBox([w_tf, w_na]),
    w_type, w_note,
    widgets.HBox([btn_save_next, btn_save, btn_skip, btn_back]),
    notice,
])
display(form)
render_current()
